# NB-05 · LangChain Deep Agent on FHIR Data
## Agentic CDI, RCM, and Patient Summary Pipeline

> **ClinIQ** | Clinical Documentation Integrity Platform

A multi-step LangChain agent equipped with 6 custom FHIR tools,
orchestrated by LangGraph, performing CDI analysis, RCM risk assessment,
and patient engagement gap detection on synthetic FHIR R4 data.

| Section | Topic |
|---------|-------|
| 1 | Setup |
| 2 | FHIR data store & synthetic bundles |
| 3 | Custom FHIR tool suite |
| 4 | LangGraph agent state machine |
| 5 | Agent scenarios |
| 6 | Reasoning trace visualisation |
| 7 | Agent evaluation |

In [1]:
import subprocess, sys

def uv_install(*packages: str) -> None:
    """Install via uv, fallback to pip."""
    try:
        subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--quiet', *packages], check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True)
        except subprocess.CalledProcessError:
            print(f'⚠️  Some packages may have failed to install, continuing with existing environment')

uv_install(
    'langchain>=0.2', 'langchain-community>=0.2',
    'langgraph>=0.1', 'langchain-core>=0.2',
    'fhir.resources>=7.1', 'pydantic>=2.5',
    'faker>=24.0', 'pandas>=2.2',
)
print('✅  Dependencies ready')

✅  Dependencies ready


In [2]:
from __future__ import annotations

import json
import logging
import random
from typing import Any, TypedDict, Annotated
import operator

from faker import Faker

SEED = 42
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s | %(levelname)-8s | %(message)s',
                    datefmt='%H:%M:%S')
log = logging.getLogger('cliniq.nb05')

## FHIR R4 Resource Hierarchy — Why It Matters for Agent Design

Understanding FHIR's resource model is critical for designing tools that a clinical agent can use reliably.

```
Patient (demographics, insurance)          ← static context
  └── Encounter (a single visit)           ← event with period
        ├── DocumentReference (note text)  ← unstructured narrative
        ├── Condition (diagnoses)           ← ICD-10 codes
        └── Procedure (what was done)       ← CPT codes
  └── Observation (lab results)            ← LOINC codes + values
  └── MedicationRequest (prescriptions)    ← RxNorm codes
```

**Why JSON string returns from tools?** LangChain's agent framework serialises all tool outputs to strings for inclusion in the LLM context window. Returning a Python dict would cause a `TypeError`. Every tool must return a string — JSON is the right format because the agent (LLM) can parse it.

**ReAct vs Chain-of-Thought:**
Chain-of-thought: reason → answer (no external grounding).
ReAct: reason → *act* (query tool) → *observe* (get real data) → reason again.
ReAct is critical for clinical agents because hallucinated lab values could be clinically dangerous. Every factual claim must be grounded in a tool call.

## § 2 · FHIR Data Store & Synthetic Bundles

In [3]:
# ── In-memory FHIR store ──────────────────────────────────────────────
ICD10_POOL = ['N18.3', 'E11.9', 'I10', 'J44.1', 'I50.9', 'F32.1']
MED_POOL   = ['metformin', 'lisinopril', 'atorvastatin', 'furosemide', 'albuterol']

def build_fhir_patient(pid: str) -> dict:
    """Build minimal FHIR R4 Patient resource."""
    return {
        'resourceType': 'Patient', 'id': pid,
        'gender': random.choice(['male', 'female']),
        'birthDate': str(fake.date_of_birth(minimum_age=40, maximum_age=85)),
        'extension': [{'url': 'insurance', 'valueString': random.choice(['Medicare', 'Medicaid', 'Commercial'])}],
    }

def build_fhir_encounter(pid: str, eid: str) -> dict:
    """Build FHIR R4 Encounter + DocumentReference bundle."""
    codes = random.sample(ICD10_POOL, k=random.randint(1, 3))
    note  = (f'Patient with {random.choice(["hypertension", "diabetes", "CKD", "COPD"])} '
             f'presenting with {random.choice(["shortness of breath", "chest pain", "fatigue", "oedema"])}. '
             f'BP {random.randint(110, 170)}/{random.randint(60, 100)}. '
             f'Plan: {random.choice(["adjust medications", "admit for observation", "refer to specialist", "increase diuretic dose"])}.') 
    return {
        'resourceType': 'Bundle', 'type': 'collection',
        'entry': [
            {'resource': {
                'resourceType': 'Encounter', 'id': eid,
                'subject': {'reference': f'Patient/{pid}'},
                'reasonCode': [{'coding': [{'code': c}]} for c in codes],
                'period': {'start': str(fake.date_this_year()), 'end': str(fake.date_this_year())},
            }},
            {'resource': {
                'resourceType': 'DocumentReference', 'id': f'DR-{eid}',
                'subject': {'reference': f'Patient/{pid}'},
                'content': [{'attachment': {'data': note}}],
            }},
        ]
    }

def build_fhir_observations(pid: str) -> list[dict]:
    """Build synthetic lab observation resources."""
    labs = {'creatinine': round(random.uniform(0.8, 6.0), 2),
            'glucose':    round(random.uniform(80, 350), 1),
            'hba1c':      round(random.uniform(5.0, 13.0), 1),}
    return [{'resourceType': 'Observation', 'subject': {'reference': f'Patient/{pid}'},
             'code': {'coding': [{'display': k}]},
             'valueQuantity': {'value': v}} for k, v in labs.items()]

# Build store with 10 synthetic patients
FHIR_STORE: dict[str, Any] = {'patients': {}, 'encounters': {}, 'observations': {}}
for i in range(1, 11):
    pid = f'P{i:03d}'
    eid = f'E{i:04d}'
    FHIR_STORE['patients'][pid]      = build_fhir_patient(pid)
    FHIR_STORE['encounters'][eid]    = build_fhir_encounter(pid, eid)
    FHIR_STORE['observations'][pid]  = build_fhir_observations(pid)

log.info('FHIR store built: %d patients', len(FHIR_STORE['patients']))

23:29:56 | INFO     | FHIR store built: 10 patients


## § 3 · Custom FHIR Tool Suite

In [4]:
from langchain.tools import tool
from langchain_core.tools import BaseTool

# ── Tool 1: Patient Summary ────────────────────────────────────────────
@tool
def get_patient_summary(patient_id: str) -> str:
    """Get demographic summary and active conditions for a patient.
    
    Args:
        patient_id: FHIR Patient resource ID (e.g. P001)
    """
    patient = FHIR_STORE['patients'].get(patient_id)
    if not patient:
        return f'Patient {patient_id} not found in store.'
    return json.dumps({
        'patient_id': patient_id,
        'gender':     patient['gender'],
        'birthDate':  patient['birthDate'],
        'insurance':  patient.get('extension', [{}])[0].get('valueString', 'Unknown'),
    }, indent=2)


# ── Tool 2: Encounter Note ─────────────────────────────────────────────
@tool
def get_encounter_note(encounter_id: str) -> str:
    """Retrieve the clinical narrative note for an encounter.
    
    Args:
        encounter_id: FHIR Encounter resource ID (e.g. E0001)
    """
    bundle = FHIR_STORE['encounters'].get(encounter_id)
    if not bundle:
        return f'Encounter {encounter_id} not found.'
    for entry in bundle.get('entry', []):
        res = entry.get('resource', {})
        if res.get('resourceType') == 'DocumentReference':
            content = res.get('content', [{}])[0]
            return content.get('attachment', {}).get('data', 'No note available.')
    return 'No DocumentReference found in encounter bundle.'


# ── Tool 3: Lab Results ───────────────────────────────────────────────
@tool
def get_lab_results(patient_id: str) -> str:
    """Retrieve recent laboratory results for a patient.
    
    Args:
        patient_id: FHIR Patient resource ID
    """
    obs = FHIR_STORE['observations'].get(patient_id, [])
    if not obs:
        return f'No observations found for {patient_id}.'
    results = []
    for o in obs:
        name = o.get('code', {}).get('coding', [{}])[0].get('display', 'Unknown')
        val  = o.get('valueQuantity', {}).get('value', 'N/A')
        results.append(f'{name}: {val}')
    return json.dumps({'patient_id': patient_id, 'labs': results}, indent=2)


# ── Tool 4: ICD-10 Search (RAG stub) ─────────────────────────────────
ICD10_MINI_DB = {
    'N18.3': 'Chronic kidney disease, stage 3 (moderate)',
    'E11.9': 'Type 2 diabetes mellitus without complications',
    'I10':   'Essential (primary) hypertension',
    'I50.9': 'Heart failure, unspecified',
    'J44.1': 'COPD with acute exacerbation',
}
@tool
def search_icd10_codes(query: str) -> str:
    """Search ICD-10-CM codes relevant to a clinical query.
    
    Args:
        query: Clinical term or phrase to search
    """
    query_lower = query.lower()
    matches = {code: desc for code, desc in ICD10_MINI_DB.items()
               if any(w in desc.lower() for w in query_lower.split())}
    return json.dumps(matches or {'message': f'No exact matches for: {query}'}, indent=2)


# ── Tool 5: CDI Gap Detector ──────────────────────────────────────────
CDI_RULES = {
    'N18':   'Missing stage qualifier (1-5 or ESRD) for CKD code',
    'E11':   'Missing complication type for Type 2 diabetes',
    'I50':   'Missing systolic/diastolic specification for heart failure',
}
@tool
def detect_cdi_gaps(icd10_codes: str) -> str:
    """Detect documentation gaps for a list of ICD-10 codes.
    
    Args:
        icd10_codes: Comma-separated ICD-10 code list e.g. 'N18,E11.9,I10'
    """
    codes = [c.strip() for c in icd10_codes.split(',')]
    gaps  = []
    for code in codes:
        chapter = code.split('.')[0]
        if chapter in CDI_RULES:
            gaps.append({'code': code, 'gap': CDI_RULES[chapter]})
    return json.dumps({'gaps': gaps, 'total_gaps': len(gaps)}, indent=2)


# ── Tool 6: Encounter Reason Codes ───────────────────────────────────
@tool
def get_encounter_codes(encounter_id: str) -> str:
    """Extract ICD-10 reason codes from an encounter.
    
    Args:
        encounter_id: FHIR Encounter ID
    """
    bundle = FHIR_STORE['encounters'].get(encounter_id)
    if not bundle:
        return f'Encounter {encounter_id} not found.'
    for entry in bundle.get('entry', []):
        res = entry.get('resource', {})
        if res.get('resourceType') == 'Encounter':
            codes = [
                rc['coding'][0]['code']
                for rc in res.get('reasonCode', [])
                if rc.get('coding')
            ]
            return json.dumps({'encounter_id': encounter_id, 'reason_codes': codes})
    return 'No codes found.'

FHIR_TOOLS = [
    get_patient_summary, get_encounter_note, get_lab_results,
    search_icd10_codes, detect_cdi_gaps, get_encounter_codes,
]
log.info('FHIR tool suite registered: %d tools', len(FHIR_TOOLS))

C:\Users\prave\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


23:30:00 | INFO     | FHIR tool suite registered: 6 tools


## § 4 · LangGraph Agent State Machine

In [5]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END

class AgentState(TypedDict):
    """Typed state passed between LangGraph nodes."""
    input:        str
    patient_id:   str
    encounter_id: str
    tool_calls:   Annotated[list, operator.add]   # reducer: accumulate across nodes
    findings:     dict
    final_report: str
    step_count:   int

# ── Define node functions ─────────────────────────────────────────────
def planner_node(state: AgentState) -> AgentState:
    """Extract patient/encounter IDs from input query."""
    inp = state['input']
    pid = next((w for w in inp.split() if w.startswith('P') and w[1:].isdigit()), 'P001')
    eid = next((w for w in inp.split() if w.startswith('E') and w[1:].isdigit()), 'E0001')
    log.info('[PLANNER] Routing to patient=%s encounter=%s', pid, eid)
    return {**state, 'patient_id': pid, 'encounter_id': eid, 'step_count': 0}

def cdi_analyser_node(state: AgentState) -> AgentState:
    """Run CDI gap analysis on the encounter."""
    codes_raw = get_encounter_codes.invoke(state['encounter_id'])
    codes     = json.loads(codes_raw).get('reason_codes', [])
    gaps_raw  = detect_cdi_gaps.invoke(','.join(codes)) if codes else '{"gaps":[]}'
    gaps      = json.loads(gaps_raw)
    note      = get_encounter_note.invoke(state['encounter_id'])
    findings  = {**state.get('findings', {}), 'cdi_gaps': gaps, 'note_snippet': note[:100]}
    tool_log  = [f'get_encounter_codes({state["encounter_id"]})',
                 f'detect_cdi_gaps({codes})',
                 f'get_encounter_note({state["encounter_id"]})']
    log.info('[CDI_ANALYSER] %d gaps found', gaps['total_gaps'])
    return {**state, 'findings': findings, 'tool_calls': tool_log,
            'step_count': state['step_count'] + 1}

def report_writer_node(state: AgentState) -> AgentState:
    """Synthesise findings into a structured report."""
    gaps = state['findings'].get('cdi_gaps', {})
    note = state['findings'].get('note_snippet', '')
    report_lines = [
        f'CDI REPORT | Patient: {state["patient_id"]} | Encounter: {state["encounter_id"]}',
        f'Note excerpt: "{note}..."',
        f'Documentation gaps: {gaps.get("total_gaps", 0)}',
    ]
    for g in gaps.get('gaps', []):
        report_lines.append(f'  • [{g["code"]}] {g["gap"]}')
    report = '\n'.join(report_lines)
    log.info('[REPORT_WRITER] Report generated (%d lines)', len(report_lines))
    return {**state, 'final_report': report}

def should_continue(state: AgentState) -> str:
    """Conditional edge: stop if max steps reached, otherwise continue."""
    if state['step_count'] >= 3:
        return 'write_report'
    return 'analyse_cdi'

# ── Build LangGraph ───────────────────────────────────────────────────
workflow = StateGraph(AgentState)
workflow.add_node('plan',         planner_node)
workflow.add_node('analyse_cdi',  cdi_analyser_node)
workflow.add_node('write_report', report_writer_node)

workflow.set_entry_point('plan')
workflow.add_edge('plan', 'analyse_cdi')
workflow.add_conditional_edges('analyse_cdi', should_continue,
                                {'analyse_cdi': 'analyse_cdi', 'write_report': 'write_report'})
workflow.add_edge('write_report', END)

graph = workflow.compile()
log.info('✅  LangGraph compiled: %d nodes', len(workflow.nodes))
print('Nodes:', list(workflow.nodes.keys()))

23:30:00 | INFO     | ✅  LangGraph compiled: 3 nodes


Nodes: ['plan', 'analyse_cdi', 'write_report']


### LangGraph State Machine — Visual Structure

```
┌─────────────┐       ┌────────────────┐
│   START     │──────▶│     plan       │
│  (entry)    │       │ (extract IDs)  │
└─────────────┘       └───────┬────────┘
                               │
                               ▼
                    ┌────────────────────┐
                    │   analyse_cdi      │◀──────┐
                    │ (tools + findings) │       │
                    └─────────┬──────────┘       │
                              │                   │
                    conditional_edge:             │
                    step_count < 3 ?──────── YES──┘
                              │ NO
                              ▼
                    ┌────────────────────┐
                    │   write_report     │
                    │  (synthesise)      │
                    └─────────┬──────────┘
                              │
                              ▼
                           [ END ]
```

**Why LangGraph over simple LangChain chains?**
- Chains are linear (A→B→C). Agents need **loops** (re-analyse if gaps found, retry on tool error)
- LangGraph makes the control flow **explicit and type-safe** via TypedDict state
- The `Annotated[list, operator.add]` reducer merges tool_calls across all node invocations — crucial for audit trails
- State checkpointing allows **resumability** — if the agent fails mid-pipeline, it can restart from the last checkpoint

## § 5 · Agent Scenarios

In [6]:
# ── Scenario A: CDI Analysis via LangGraph ───────────────────────────
print('\n' + '='*60)
print('SCENARIO A — CDI Gap Analysis (LangGraph)')
print('='*60)

state_a = graph.invoke({
    'input':        'Analyse patient P001 encounter E0001 for CDI gaps',
    'patient_id':   'P001',
    'encounter_id': 'E0001',
    'tool_calls':   [],
    'findings':     {},
    'final_report': '',
    'step_count':   0,
})
print('\n' + state_a['final_report'])
print(f'\nTool calls: {state_a["tool_calls"]}')
print(f'Steps taken: {state_a["step_count"]}')

23:30:00 | INFO     | [PLANNER] Routing to patient=P001 encounter=E0001


23:30:00 | INFO     | [CDI_ANALYSER] 2 gaps found


23:30:00 | INFO     | [CDI_ANALYSER] 2 gaps found


23:30:00 | INFO     | [CDI_ANALYSER] 2 gaps found


23:30:00 | INFO     | [REPORT_WRITER] Report generated (5 lines)



SCENARIO A — CDI Gap Analysis (LangGraph)

CDI REPORT | Patient: P001 | Encounter: E0001
Note excerpt: "Patient with diabetes presenting with shortness of breath. BP 153/94. Plan: adjust medications...."
Documentation gaps: 2
  • [E11.9] Missing complication type for Type 2 diabetes
  • [I50.9] Missing systolic/diastolic specification for heart failure

Tool calls: ['get_encounter_codes(E0001)', "detect_cdi_gaps(['I10', 'E11.9', 'I50.9'])", 'get_encounter_note(E0001)', 'get_encounter_codes(E0001)', "detect_cdi_gaps(['I10', 'E11.9', 'I50.9'])", 'get_encounter_note(E0001)', 'get_encounter_codes(E0001)', "detect_cdi_gaps(['I10', 'E11.9', 'I50.9'])", 'get_encounter_note(E0001)', 'get_encounter_codes(E0001)', "detect_cdi_gaps(['I10', 'E11.9', 'I50.9'])", 'get_encounter_note(E0001)', 'get_encounter_codes(E0001)', "detect_cdi_gaps(['I10', 'E11.9', 'I50.9'])", 'get_encounter_note(E0001)', 'get_encounter_codes(E0001)', "detect_cdi_gaps(['I10', 'E11.9', 'I50.9'])", 'get_encounter_note(E0001)']


In [7]:
# ── Scenario B: Patient engagement summary ───────────────────────────
print('\n' + '='*60)
print('SCENARIO B — Patient Engagement Gap Summary')
print('='*60)

def run_engagement_scenario(pid: str) -> str:
    """Identify care gaps for patient engagement outreach."""
    summary = json.loads(get_patient_summary.invoke(pid))
    labs    = json.loads(get_lab_results.invoke(pid))

    care_gaps: list[str] = []
    lab_vals  = {l.split(':')[0].strip(): float(l.split(':')[1]) for l in labs.get('labs', []) if ':' in l}

    if lab_vals.get('hba1c', 0) > 7.0:
        care_gaps.append('HbA1c above target — diabetes education referral recommended')
    if lab_vals.get('creatinine', 0) > 2.0:
        care_gaps.append('Elevated creatinine — nephrology follow-up recommended')
    if lab_vals.get('glucose', 0) > 200:
        care_gaps.append('Hyperglycaemia — medication review needed')

    return (
        f'ENGAGEMENT SUMMARY\nPatient: {pid} | Insurance: {summary["insurance"]}\n'
        f'Care gaps identified: {len(care_gaps)}\n'
        + '\n'.join(f'  - {g}' for g in care_gaps)
    )

print(run_engagement_scenario('P002'))


SCENARIO B — Patient Engagement Gap Summary
ENGAGEMENT SUMMARY
Patient: P002 | Insurance: Commercial
Care gaps identified: 2
  - HbA1c above target — diabetes education referral recommended
  - Elevated creatinine — nephrology follow-up recommended


In [8]:
# ── Scenario C: RCM Risk Assessment via LangGraph ────────────────────
print('\n' + '='*60)
print('SCENARIO C — RCM Risk Assessment (LangGraph)')
print('='*60)

initial_state: AgentState = {
    'input':        'Analyse patient P003 encounter E0003 for RCM risk',
    'patient_id':   '',
    'encounter_id': '',
    'tool_calls':   [],
    'findings':     {},
    'final_report': '',
    'step_count':   0,
}

final_state = graph.invoke(initial_state)
print('\n' + final_state['final_report'])
print(f'\nTool calls made: {final_state["tool_calls"]}')

23:30:00 | INFO     | [PLANNER] Routing to patient=P003 encounter=E0003


23:30:00 | INFO     | [CDI_ANALYSER] 1 gaps found


23:30:00 | INFO     | [CDI_ANALYSER] 1 gaps found


23:30:00 | INFO     | [CDI_ANALYSER] 1 gaps found


23:30:00 | INFO     | [REPORT_WRITER] Report generated (4 lines)



SCENARIO C — RCM Risk Assessment (LangGraph)

CDI REPORT | Patient: P003 | Encounter: E0003
Note excerpt: "Patient with CKD presenting with shortness of breath. BP 115/84. Plan: adjust medications...."
Documentation gaps: 1
  • [E11.9] Missing complication type for Type 2 diabetes

Tool calls made: ['get_encounter_codes(E0003)', "detect_cdi_gaps(['E11.9', 'F32.1'])", 'get_encounter_note(E0003)', 'get_encounter_codes(E0003)', "detect_cdi_gaps(['E11.9', 'F32.1'])", 'get_encounter_note(E0003)', 'get_encounter_codes(E0003)', "detect_cdi_gaps(['E11.9', 'F32.1'])", 'get_encounter_note(E0003)', 'get_encounter_codes(E0003)', "detect_cdi_gaps(['E11.9', 'F32.1'])", 'get_encounter_note(E0003)', 'get_encounter_codes(E0003)', "detect_cdi_gaps(['E11.9', 'F32.1'])", 'get_encounter_note(E0003)', 'get_encounter_codes(E0003)', "detect_cdi_gaps(['E11.9', 'F32.1'])", 'get_encounter_note(E0003)']


In [9]:
# ── Token counting — context window management ──────────────────────
# In production, clinical notes can be 2000+ tokens. Managing context is critical.
# Claude Sonnet 4 has a 200k token window; smaller models (Qwen 1.5B) ~32k.

def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token (GPT-style BPE approximation)."""
    return len(text) // 4

# Simulate accumulating tool call results over a multi-step agent run
tool_results_accumulated = []
for pid in ['P001', 'P002', 'P003']:
    summary = get_patient_summary.invoke(pid)
    labs    = get_lab_results.invoke(pid)
    tool_results_accumulated.extend([summary, labs])

total_chars  = sum(len(r) for r in tool_results_accumulated)
total_tokens = total_chars // 4   # ~4 chars per token

print('\n📊 Context Window Budget Analysis')
print(f'  Tool results accumulated: {len(tool_results_accumulated)} calls')
print(f'  Total chars: {total_chars:,} | Estimated tokens: {total_tokens:,}')
print()

# Context budget for different models
MODELS = {
    'Qwen2.5-1.5B-Instruct': 32_768,
    'Claude Sonnet 4.6':    200_000,
    'GPT-4o':               128_000,
}
for model, ctx_limit in MODELS.items():
    budget_used = total_tokens / ctx_limit
    status = '⚠️  over budget' if budget_used > 0.8 else '✅  within budget'
    print(f'  {model:<30} {budget_used:.1%} of {ctx_limit//1000}k  {status}')

print()
print('💡 Production strategy:')
print('   1. Summarise tool results before adding to context (compress long notes)')
print('   2. Use sliding window: keep only last N tool calls in context')
print('   3. Retrieve only patient-specific context, not entire knowledge base')


📊 Context Window Budget Analysis
  Tool results accumulated: 6 calls
  Total chars: 627 | Estimated tokens: 156

  Qwen2.5-1.5B-Instruct          0.5% of 32k  ✅  within budget
  Claude Sonnet 4.6              0.1% of 200k  ✅  within budget
  GPT-4o                         0.1% of 128k  ✅  within budget

💡 Production strategy:
   1. Summarise tool results before adding to context (compress long notes)
   2. Use sliding window: keep only last N tool calls in context
   3. Retrieve only patient-specific context, not entire knowledge base


## § 6 · Reasoning Trace Visualisation

In [10]:
from IPython.display import HTML, display

STEP_COLOURS = {
    'thought':     ('#E0F2F1', '#0E7C7B'),
    'action':      ('#FEF3C7', '#D97706'),
    'observation': ('#F3F4F6', '#6B7280'),
}

def render_trace(trace: list[dict]) -> None:
    """Render agent ReAct trace as coloured HTML in the notebook."""
    html_parts = ['<div style="font-family: monospace; font-size: 13px;">']
    for i, step in enumerate(trace, 1):
        bg, fg = STEP_COLOURS.get(step['type'], ('#fff', '#000'))
        label  = step['type'].upper()
        if step['type'] == 'action':
            content = f"<b>{step['tool']}</b>({step['input'][:80]})"
        else:
            content = step.get('content', '')[:200]
        html_parts.append(
            f'<div style="background:{bg};color:{fg};border-left:4px solid {fg};'
            f'padding:8px 12px;margin:4px 0;border-radius:4px;">'
            f'<span style="font-weight:bold;font-size:11px;">[{i}] {label}</span><br/>'
            f'{content}</div>'
        )
    html_parts.append('</div>')
    display(HTML(''.join(html_parts)))

# Build trace from LangGraph Scenario A results
scenario_a_trace = [
    {'type': 'thought', 'content': f'Planning: extract patient/encounter IDs from input query'},
    {'type': 'action', 'tool': 'get_encounter_codes', 'input': state_a['encounter_id']},
    {'type': 'observation', 'content': f'Codes found for encounter {state_a["encounter_id"]}'},
    {'type': 'action', 'tool': 'detect_cdi_gaps', 'input': str(state_a.get('findings', {}).get('cdi_gaps', {}))},
    {'type': 'observation', 'content': f'CDI gaps: {state_a.get("findings", {}).get("cdi_gaps", {}).get("total_gaps", 0)} found'},
    {'type': 'action', 'tool': 'get_encounter_note', 'input': state_a['encounter_id']},
    {'type': 'thought', 'content': 'Synthesising findings into structured CDI report'},
]

print('\n📋 Agent Reasoning Trace (Scenario A):')
render_trace(scenario_a_trace)


📋 Agent Reasoning Trace (Scenario A):


## § 7 · Agent Evaluation

In [11]:
# ── Gold standard QA pairs (expanded to 6 representative cases) ───────
GOLD_QA: list[dict] = [
    {'patient_id': 'P001', 'encounter_id': 'E0001',
     'question': 'Are there CDI gaps for encounter E0001?',
     'gold_answer_contains': ['gap', 'documentation']},
    {'patient_id': 'P002', 'encounter_id': None,
     'question': 'What are the care gaps for patient P002?',
     'gold_answer_contains': ['care gaps', 'recommended']},
    {'patient_id': 'P003', 'encounter_id': 'E0003',
     'question': 'What codes were assigned to encounter E0003?',
     'gold_answer_contains': ['code', 'reason']},
    {'patient_id': 'P004', 'encounter_id': None,
     'question': 'What labs are elevated for patient P004?',
     'gold_answer_contains': ['lab', 'creatinine', 'glucose', 'hba1c']},
    {'patient_id': 'P005', 'encounter_id': 'E0005',
     'question': 'Does encounter E0005 have missing stage qualifiers?',
     'gold_answer_contains': ['stage', 'qualifier', 'gap']},
    {'patient_id': 'P006', 'encounter_id': None,
     'question': 'Is patient P006 at risk for readmission based on labs?',
     'gold_answer_contains': ['risk', 'care', 'recommended']},
]

def evaluate_agent_output(output: str, gold_contains: list[str]) -> float:
    """Keyword containment faithfulness score — proxy for answer correctness."""
    output_lower = output.lower()
    hits = sum(1 for kw in gold_contains if kw.lower() in output_lower)
    return hits / len(gold_contains)

def run_scenario(qa: dict) -> str:
    if qa['encounter_id']:
        state = graph.invoke({
            'input': qa['question'], 'patient_id': qa['patient_id'],
            'encounter_id': qa['encounter_id'], 'tool_calls': [],
            'findings': {}, 'final_report': '', 'step_count': 0,
        })
        return state['final_report']
    else:
        return run_engagement_scenario(qa['patient_id'])

print('\n📊 Agent Evaluation — 6 Gold QA Pairs')
print('─' * 50)
scores = []
for qa in GOLD_QA:
    try:
        out   = run_scenario(qa)
        score = evaluate_agent_output(out, qa['gold_answer_contains'])
    except Exception as e:
        out, score = str(e), 0.0
    scores.append(score)
    status = '✅' if score >= 0.5 else '⚠️'
    print(f'  {status} Q: "{qa["question"][:50]}..." → faithfulness: {score:.2f}')
print(f'\n  Mean faithfulness: {sum(scores)/len(scores):.3f}')
print(f'  Pass rate (≥0.5): {sum(1 for s in scores if s>=0.5)}/{len(scores)}')
print('\n💡 In production: replace graph nodes with Qwen2.5-1.5B-Instruct tool-calling agent')

23:30:01 | INFO     | [PLANNER] Routing to patient=P001 encounter=E0001


23:30:01 | INFO     | [CDI_ANALYSER] 2 gaps found


23:30:01 | INFO     | [CDI_ANALYSER] 2 gaps found


23:30:01 | INFO     | [CDI_ANALYSER] 2 gaps found


23:30:01 | INFO     | [REPORT_WRITER] Report generated (5 lines)


...

23:30:01 | INFO     | [CDI_ANALYSER] 1 gaps found


23:30:01 | INFO     | [CDI_ANALYSER] 1 gaps found


23:30:01 | INFO     | [CDI_ANALYSER] 1 gaps found


23:30:01 | INFO     | [REPORT_WRITER] Report generated (4 lines)



📊 Agent Evaluation — 6 Gold QA Pairs
──────────────────────────────────────────────────
  ✅ Q: "Are there CDI gaps for encounter E0001?..." → faithfulness: 1.00
  ✅ Q: "What are the care gaps for patient P002?..." → faithfulness: 1.00
  ⚠️ Q: "What codes were assigned to encounter E0003?..." → faithfulness: 0.00
  ⚠️ Q: "What labs are elevated for patient P004?..." → faithfulness: 0.25
  ⚠️ Q: "Does encounter E0005 have missing stage qualifiers..." → faithfulness: 0.33
  ✅ Q: "Is patient P006 at risk for readmission based on l..." → faithfulness: 0.67

  Mean faithfulness: 0.542
  Pass rate (≥0.5): 3/6

💡 In production: replace graph nodes with Qwen2.5-1.5B-Instruct tool-calling agent


In [12]:
# ── Graceful degradation — tool error handling ──────────────────────
# Production agents MUST handle tool failures gracefully.
# A clinical agent that crashes on a missing record is unsafe.

def safe_tool_invoke(tool_fn, input_val: str, fallback: str = '{}') -> str:
    """Invoke a LangChain tool with error handling and structured fallback."""
    try:
        result = tool_fn.invoke(input_val)
        return result if result else fallback
    except Exception as e:
        log.warning('Tool %s failed on input %r: %s', tool_fn.name, input_val[:50], e)
        return json.dumps({'error': str(e), 'tool': tool_fn.name, 'input': input_val})

print('\n📊 Graceful Degradation Demo')
print('─' * 45)
test_cases = [
    (get_patient_summary, 'P001'),          # valid
    (get_patient_summary, 'P999'),          # patient not found
    (get_encounter_note,  'E_INVALID'),     # encounter not found
    (get_lab_results,     'P001'),          # valid
]
for fn, inp in test_cases:
    result = safe_tool_invoke(fn, inp)
    parsed = json.loads(result) if result.startswith('{') else {'raw': result[:80]}
    has_error = 'error' in parsed or 'not found' in result.lower()
    status = '⚠️  graceful error' if has_error else '✅  success'
    print(f'  {fn.name}({inp:<12}) → {status}')

print()
print('💡 The agent should interpret error responses and decide:')
print('   - Missing patient: ask user to verify ID')
print('   - Missing encounter: use latest available encounter instead')
print('   - Tool timeout: retry once then flag for human review')


📊 Graceful Degradation Demo
─────────────────────────────────────────────
  get_patient_summary(P001        ) → ✅  success
  get_patient_summary(P999        ) → ⚠️  graceful error
  get_encounter_note(E_INVALID   ) → ⚠️  graceful error
  get_lab_results(P001        ) → ✅  success

💡 The agent should interpret error responses and decide:
   - Missing patient: ask user to verify ID
   - Missing encounter: use latest available encounter instead
   - Tool timeout: retry once then flag for human review
